In [ ]:
import folium
import pandas as pd
import geopandas as gpd
from glob import glob
from os.path import join
import branca.colormap as cm
from folium.plugins import TimeSliderChoropleth

In [ ]:
data_path = ""
path_zips = "/glade/work/cbecker/california_zip.geojson"

In [ ]:
df = pd.read_parquet(data_path, engine='pyarrow')
dt_idx = pd.to_datetime([f"2024-03-{date}" for date in df.index.levels[1].values])
dt_idx = dt_idx.astype("int64") // 10 ** 9
df.index = df.index.set_levels(dt_idx.astype("U10"), level=1)
df

In [ ]:
gdf_zips = gpd.read_file(path_zips)
gdf_zips.set_index('ZIP_CODE', inplace=True)
gdf_zips.drop('OBJECTID', axis=1, inplace=True)
gdf_zips

In [ ]:
min_color = df.min().min()
max_color = df.max().max()
cmap = cm.linear.PuRd_09.scale(min_color, max_color)
df_style = df.map(cmap, na_action="ignore").rename(columns={"radius_of_gyration": "color"})
df_style = df_style.fillna(0, downcast="int64")
df_style["opacity"] = 1.0

In [ ]:
styledict = {}
for zip in df_style.index.levels[0]:
    styledict[zip] = df_style.loc[zip].to_dict(orient="index")

In [ ]:
m = folium.Map(location=[34.0549, -118.2426],
               zoom_start=8)

TimeSliderChoropleth(
    gdf_zips.to_json(),
    styledict=styledict,
).add_to(m)

m.save("radiusofgyration_byzip_bytime.html")
m

![title](img/radgyr_byzip_bytime.png)